In [ ]:
from pymodulon.core import IcaData
from pymodulon.plotting import *
from os import path
import pandas as pd
import re
from Bio.KEGG import REST
from tqdm.notebook import tqdm

In [ ]:
from pymodulon.compare import *
from pymodulon.io import *

In [ ]:
ica_data_dir = '../data/ica_runs_prot/ica_runs/100/'

## Load data and create ICA object

#### Metadata

In [ ]:
df_metadata = pd.read_csv('../data/processed_data/metadata.tsv',index_col=0, sep='\t')
display(
    df_metadata.head(),
    df_metadata.shape
)

In [ ]:
print(df_metadata.project.notnull().all())
print(df_metadata.condition.notnull().all())

#### ICA Data

In [ ]:
A=pd.read_csv(path.join(ica_data_dir,'A.csv'),index_col=0)
M=pd.read_csv(path.join(ica_data_dir,'M.csv'),index_col=0)
X=pd.read_csv('../data/processed_data/log_normalizedCounts_norm.csv',index_col=0)
set(X.columns)-set(A.columns)
A[X.columns].to_csv(path.join(ica_data_dir,'A.csv'))

In [ ]:
X_log_tpm=pd.read_csv('../data/processed_data/log_normalizedCounts.csv',index_col=0)
set(X.columns)-set(X_log_tpm.columns)
X_log_tpm = X_log_tpm[X.columns]

In [ ]:
for matrix in M, X, X_log_tpm:
    for index in matrix.index: # remove 'gene-' from each gene 
        matrix.rename(index={index:index.strip('gene-')},inplace=True)

##### TRN

In [ ]:
trn = pd.read_csv('../data/external/trn.csv', index_col=0).dropna()
trn = trn[["reg","gene_name", "gene_id", "effect"]]
trn = trn.rename({"reg":"regulator"}, axis=1)


count = 0
genes_in_regulons = trn["gene_id"].unique()
for gene in X.index:
    if gene in genes_in_regulons:
        count+=1
print("Number of genes with annotated regulators: ",count)

#filter trn to only include genes which were aligned to
trn

In [ ]:
print(trn.regulator.notnull().all())
print(trn.gene_id.notnull().all())

In [ ]:
chr_info = pd.read_csv('../data/processed_data/chromosome_info.csv', index_col=0)

In [ ]:
#M matrix renamed as S matrix

ica_data = IcaData(M = M,
                   A = A,
                   X = X,
                   log_tpm = X_log_tpm,
                   gene_table = '../data/processed_data/gene_info.csv',
                   sample_table = '../data/processed_data/metadata.tsv',
                   trn = trn[trn.gene_id.isin(X.index.to_list())],
                   chrom=chr_info
                  )

In [ ]:
# uncomment to reoptimize thresholds
ica_data.reoptimize_thresholds()

In [ ]:
# set value for optimal threshold after it has been calculated the first time
# ica_data.recompute_thresholds(500)

In [ ]:
from pymodulon.util import explained_variance
explained_variance(ica_data)

In [ ]:
# add individual explained variance for each iModulon

for k in ica_data.imodulon_table.index:
    ica_data.imodulon_table.loc[k, 'exp_var'] = explained_variance(
        ica_data, imodulons=k)

# Adjust Thresholds

In [ ]:
# for iModulons with no genes based on kurtosis metric, adjust to include top 1% of genes for enrichment purposes
for im in ica_data.imodulon_table.index:
    if len(ica_data.view_imodulon(im)) == 0:
        lenient_threshold = ica_data.M[im].abs().quantile(0.99)
        ica_data.thresholds[im] =  lenient_threshold
        ica_data.change_threshold(im, lenient_threshold)

# for iModulons with overly strict threshold or thresholding based on manual curation, adjust to include top 1% of genes for enrichment
ims_to_adjust = [73, 72, 69, 14]

for im in ims_to_adjust:
    lenient_threshold = ica_data.M[im].abs().quantile(0.99)
    ica_data.thresholds[im] =  lenient_threshold
    ica_data.change_threshold(im, lenient_threshold)


# Compute Enrichments

#### Single Gene iModulons
Determine single gene imodulons and set threshold so that they are not inaccurately enriched with regulations and functions unrelated to their primary gene due to the automatically optimized thresholding

In [ ]:
sg_imods = ica_data.find_single_gene_imodulons()
print(sg_imods)

In [ ]:
sg_imods = ica_data.find_single_gene_imodulons(save=True)

for i,mod in enumerate(sg_imods):
    new_thresh = ica_data.M[mod].max() * .9 # threshold is 90% of max of top value for SG iMods
    ica_data.thresholds[mod] =  new_thresh # chosen to mark a point at which the highest weighted gene dominates imodulon
    ica_data.change_threshold(mod, new_thresh)
    
ica_data.imodulon_table.single_gene.fillna(False, inplace=True)

    
# add imodulon size for each imodulon
for i in ica_data.imodulon_table.index:
    ica_data.imodulon_table.at[i, "imodulon_size"] = len(ica_data.view_imodulon(i))

#### Regulatory Enrichments

In [ ]:
# all intersections of 1/2 regulators present in the dataset
trn_enrichment = ica_data.compute_trn_enrichment(save=False,fdr=1e-5, max_regs=2, method='and', force=True)
trn_enrichment

In [ ]:
# save file
trn_enrichment.to_csv('../data/processed_data/trn_enrichments.csv')

In [ ]:
trn = pd.read_csv('../data/external/trn_full.csv', index_col=0).dropna()
trn = trn[["reg","gene_name", "gene_id", "effect"]]
trn = trn.rename({"reg":"regulator"}, axis=1)


count = 0
genes_in_regulons = trn["gene_id"].unique()
for gene in X.index:
    if gene in genes_in_regulons:
        count+=1
print("Number of genes with annotated regulators: ",count)

trn = trn[trn.gene_id.isin(ica_data.gene_table.index)]
#filter trn to only include genes which were aligned to
ica_data.trn = trn

In [ ]:
trn_enrichment_supp = ica_data.compute_trn_enrichment(save=False,fdr=1e-5, max_regs=1)
trn_enrichment_supp.to_csv('../data/processed_data/trn_enrichments_supp.csv')

#### KEGG Enrichments

In [ ]:
DF_KEGG = pd.read_csv('../data/sequence_files/kegg_mapping.csv',index_col=0)
print(DF_KEGG.database.unique())
DF_KEGG.head()

In [ ]:
kegg_pathways = DF_KEGG[DF_KEGG.database == 'KEGG_pathway']
kegg_modules = DF_KEGG[DF_KEGG.database == 'KEGG_module']

In [ ]:
DF_pathway_enrich = ica_data.compute_annotation_enrichment(kegg_pathways,'kegg_id')
DF_module_enrich = ica_data.compute_annotation_enrichment(kegg_modules,'kegg_id')

In [ ]:
# convert kegg pathway/module names to human readable
for idx,key in tqdm(DF_pathway_enrich.kegg_id.items(),total=len(DF_pathway_enrich)):
    text = REST.kegg_find('pathway',key).read()
    try:
        name = re.search('\t(.*)\n',text).group(1)
        DF_pathway_enrich.loc[idx,'pathway_name'] = name
    except AttributeError:
        DF_pathway_enrich.loc[idx,'pathway_name'] = None
    
for idx,key in tqdm(DF_module_enrich.kegg_id.items(),total=len(DF_module_enrich)):
    text = REST.kegg_find('module',key).read()
    try:
        name = re.search('\t(.*)\n',text).group(1)
        DF_module_enrich.loc[idx,'module_name'] = name
    except AttributeError:
        DF_module_enrich.loc[idx,'module_name'] = None

In [ ]:
DF_pathway_enrich.head()

In [ ]:
DF_module_enrich.head()

In [ ]:
# save files
DF_pathway_enrich.to_csv('../data/processed_data/kegg_pathway_enrichments.csv')
DF_module_enrich.to_csv('../data/processed_data/kegg_module_enrichments.csv')

#### GO Enrichment

In [ ]:
DF_GO = pd.DataFrame(columns=["gene_id","GO_term"])
for element in tqdm(ica_data.gene_table[~ica_data.gene_table.GO.isna()].index):
    if ica_data.gene_table.loc[element, "GO"] != float("nan"):
        terms = ica_data.gene_table.loc[element, "GO"].split(",")
        for term in terms:
            DF_GO.loc[len(DF_GO)] = [element, term]
DF_GO

In [ ]:
DF_GO.to_csv("../data/processed_data/go_data_processed.csv")

In [ ]:
DF_go_enrich = ica_data.compute_annotation_enrichment(DF_GO,'GO_term')

In [ ]:
DF_go_enrich

In [ ]:
DF_go_enrich.to_csv('../data/processed_data/go_enrichment.csv')

In [ ]:
DF_go_enrich.sort_values('qvalue').head(20)